# Spatial Inpainting Pre-training for SwinUnet

Self-supervised pre-training: mask a large spatial block (64x64 pixels = ~24km)
across ALL channels and reconstruct it from surrounding context.

This forces the model to learn:
- Terrain continuity (elevation gradients, slope patterns)
- Weather spatial structure (temperature gradients across terrain)
- Vegetation-terrain-weather relationships

**Prerequisites:**
- Pre-training TIFs in GCS bucket
- **Runtime -> Change runtime type -> GPU**

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Configuration

In [ ]:
REPO_ORG   = "amindell11"
REPO_NAME  = "wildfire-ts-swin"
REPO_BRANCH = "feature/spatial-pretrain"

GCS_BUCKET  = "lmudl-wildfire-compilation-bucket"
GCS_PREFIX  = "WildfireSpreadTS/pretrain"
LOCAL_TIF_DIR = "/content/pretrain_tifs"

OUTPUT_DIR  = "/content/drive/MyDrive/wildfire_runs/pretrain_spatial"

MAX_EPOCHS          = 200
BATCH_SIZE          = 32
BASE_LR             = 1.5e-4
WEIGHT_DECAY        = 0.05
WARMUP_EPOCHS       = 10
MASK_SIZE           = 64     # pixels (64 = half the image = ~24km)
CROP_SIDE_LENGTH    = 128
CROPS_PER_TIF       = 4
STATS_YEARS         = (2020, 2021)
NUM_WORKERS         = 4
SEED                = 42

CHECKPOINT_INTERVAL = 20

## 3. Download TIFs from GCS

In [ ]:
from google.colab import auth
auth.authenticate_user()

import os
os.makedirs(LOCAL_TIF_DIR, exist_ok=True)

!gsutil -m cp -r gs://{GCS_BUCKET}/{GCS_PREFIX}/* {LOCAL_TIF_DIR}/

import glob
tifs = glob.glob(f"{LOCAL_TIF_DIR}/**/*.tif", recursive=True)
print(f"Downloaded {len(tifs)} TIF files")

## 4. Clone repo and install dependencies

In [ ]:
_repo_url = f"https://github.com/{REPO_ORG}/{REPO_NAME}.git"
!rm -rf /content/{REPO_NAME}
!git clone -b {REPO_BRANCH} {_repo_url} /content/{REPO_NAME}
!pip install -q -r /content/{REPO_NAME}/requirements.txt
!pip install -q rasterio
!git -C /content/{REPO_NAME} log --oneline -5

In [ ]:
import sys
sys.path.insert(0, f'/content/{REPO_NAME}')

## 5. Detect accelerator

In [ ]:
import torch

if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    DEVICE = torch.device('cpu')
    print("WARNING: No GPU detected.")

## 6. Build Model & Preview Masking

In [ ]:
import os
import random
import types
import numpy as np
import torch
import torch.backends.cudnn as cudnn

from config import get_config
from networks.spatial_pretrain_swin_unet import SpatialInpaintSwinUnet
from trainer_spatial_pretrain import trainer_spatial_pretrain
from datasets.wildfire.pretrain_dataset import PretrainDataset

if torch.cuda.is_available():
    cudnn.benchmark = True
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

config_args = types.SimpleNamespace(
    cfg=f'/content/{REPO_NAME}/configs/swin_tiny_patch4_window4_128_wildfire.yaml',
    opts=['MODEL.SWIN.IN_CHANS', '40', 'MODEL.SWIN.N_TIMESTEPS', '1', 'MODEL.PRETRAIN_CKPT', 'None'],
    batch_size=None, zip=False, cache_mode='part', resume=None,
    accumulation_steps=None, use_checkpoint=False, amp_opt_level='',
    tag=None, eval=False, throughput=False,
)
config = get_config(config_args)

model = SpatialInpaintSwinUnet(
    config, img_size=CROP_SIDE_LENGTH, use_factored_embed=False,
    mask_size=MASK_SIZE,
).to(DEVICE)

dataset = PretrainDataset(
    tif_dir=LOCAL_TIF_DIR,
    crop_side_length=CROP_SIDE_LENGTH,
    stats_years=STATS_YEARS,
)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")
print(f"Mask size: {MASK_SIZE}x{MASK_SIZE} pixels ({MASK_SIZE * 375 / 1000:.0f}km x {MASK_SIZE * 375 / 1000:.0f}km)")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Preview: show the spatial mask + reconstruction from untrained model
VIS_CHANNELS = {
    'NDVI': 3,
    'Temp Max': 9,
    'Elevation': 14,
    'PDSI': 15,
    'Wind Spd': 6,
    'Fcst Temp': 36,
}

sample = dataset[0].unsqueeze(0).to(DEVICE)

model.eval()
with torch.no_grad():
    loss, pred, mask = model(sample)

x_orig = sample[0].cpu()
x_masked = (sample * (1.0 - mask))[0].cpu()
x_pred = pred[0].cpu()
mask_2d = mask[0, 0].cpu()  # (H, W)

def normalize_for_display(t):
    lo, hi = t.min(), t.max()
    if hi - lo < 1e-8:
        return torch.zeros_like(t)
    return (t - lo) / (hi - lo)

n_ch = len(VIS_CHANNELS)
fig, axes = plt.subplots(3, n_ch, figsize=(3 * n_ch, 9))

row_labels = ['Original', 'Masked Input', 'Reconstruction\n(untrained)']
for col, (name, ch) in enumerate(VIS_CHANNELS.items()):
    imgs = [
        normalize_for_display(x_orig[ch]),
        normalize_for_display(x_masked[ch]),
        normalize_for_display(x_pred[ch]),
    ]
    for row in range(3):
        ax = axes[row, col]
        ax.imshow(imgs[row].numpy(), cmap='viridis', vmin=0, vmax=1)
        # Draw red rectangle around masked region
        if row == 1:
            ys, xs = torch.where(mask_2d > 0)
            if len(ys) > 0:
                rect = mpatches.Rectangle(
                    (xs.min().item(), ys.min().item()),
                    xs.max().item() - xs.min().item(),
                    ys.max().item() - ys.min().item(),
                    linewidth=2, edgecolor='red', facecolor='none')
                ax.add_patch(rect)
        ax.set_xticks([])
        ax.set_yticks([])
        if row == 0:
            ax.set_title(name, fontsize=10)
        if col == 0:
            ax.set_ylabel(row_labels[row], fontsize=9)

plt.suptitle(f'Spatial Inpainting Preview ({MASK_SIZE}x{MASK_SIZE} block, untrained, loss={loss.item():.3f})',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

mask_pct = mask_2d.mean().item() * 100
print(f"Masked region: {MASK_SIZE}x{MASK_SIZE} = {mask_pct:.0f}% of image")
print(f"After training, the reconstruction row should match the original within the red box.")

## 7. Run Pre-training

In [ ]:
NO_RESUME = False

%load_ext tensorboard
%tensorboard --logdir {OUTPUT_DIR}/pretrain_log

args = types.SimpleNamespace(
    tif_dir=LOCAL_TIF_DIR,
    batch_size=BATCH_SIZE,
    max_epochs=MAX_EPOCHS,
    base_lr=BASE_LR,
    weight_decay=WEIGHT_DECAY,
    betas=(0.9, 0.95),
    warmup_epochs=WARMUP_EPOCHS,
    num_workers=NUM_WORKERS,
    crop_side_length=CROP_SIDE_LENGTH,
    stats_years=STATS_YEARS,
    checkpoint_interval=CHECKPOINT_INTERVAL,
    no_resume=NO_RESUME,
    crops_per_tif=CROPS_PER_TIF,
)

os.makedirs(OUTPUT_DIR, exist_ok=True)
trainer_spatial_pretrain(args, model, OUTPUT_DIR, device=DEVICE)

## 8. Next steps

Pre-trained weights are saved to Google Drive:
- `{OUTPUT_DIR}/pretrain_best.pth` -- best val loss
- `{OUTPUT_DIR}/pretrain_encoder_decoder.pth` -- final epoch

To fine-tune on fire dataset, open `train_and_test_colab.ipynb` and set:
```python
PRETRAIN_WEIGHTS = "/content/drive/MyDrive/wildfire_runs/pretrain_spatial/pretrain_best.pth"
```